## Import Libraries and Configuration Settings


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
# !pip install imblearn  # Sudah terinstal di .venv

from src.evaluation import (
    compare_models,
    cross_validate_configs,
    generate_submission,
    save_and_verify_model,
    split_train_validation,
    train_and_evaluate,
    verify_smote_order,
)
from src.models import (
    DEFAULT_DTL_CONFIGS,
    DEFAULT_KNN_CONFIGS,
    DEFAULT_LOGREG_CONFIGS,
    DEFAULT_NB_CONFIGS,
    C45DecisionTree,
    KNN,
    LogisticRegression,
    WideNaiveBayes,
    get_knn_preprocessing_config,
)
from src.preprocess.dtl_features import DTL_CATEGORICAL_COLUMNS, prepare_dtl_features
from src.preprocess.nb_features import prepare_nb_features
from src.preprocess.preprocess import PreprocessingConfig, run_preprocessing

RANDOM_STATE = 824


## Data Understanding


In [ ]:
# Resolusi lokasi folder data/ fleksibel
DATA_DIR = Path('data') if Path('data').is_dir() else Path('../data')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Format submisi:\n{sample_submission.head()}')
print(f'Panjang train dataset = {len(train_df)}')
print(f'Panjang test dataset = {len(test_df)}')

train_df.describe()


In [ ]:
train_df.hist(bins=50, figsize=(20, 15))
plt.show()


In [ ]:
print('Standard Correlation Coefficient (Pearson\'s r)')
numeric_df = train_df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.show()


## Data Preprocessing (Modular Pipeline)


In [ ]:
prep = run_preprocessing(
    train_df,
    test_df,
    config=get_knn_preprocessing_config(random_state=RANDOM_STATE, verbose=True),
)


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(prep.cumulative_variance) + 1), prep.cumulative_variance, marker='o', linestyle='--')
plt.axhline(y=0.95, color='r', linestyle='-', label='95% Explained Variance')
plt.title('Cumulative Explained Variance by PCA Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Variance Explained')
plt.legend()
plt.grid(True)
plt.show()


## Modeling, Validation, and Model Comparison


In [ ]:
verify_smote_order(prep.X_train_encoded, prep.X_train_balanced, verbose=True)
split_data = split_train_validation(
    prep.X_train,
    prep.y_train,
    is_orig=prep.is_orig,
    test_size=0.15,
    random_state=RANDOM_STATE,
    verbose=True,
)

evaluation_results = []


### Model 1: K-Nearest Neighbors (KNN Scratch)


In [ ]:
best_knn_config, knn_cv_df = cross_validate_configs(
    KNN,
    DEFAULT_KNN_CONFIGS,
    split_data.X_tr,
    split_data.y_tr,
    k_folds=5,
    random_state=RANDOM_STATE,
    model_name="KNN",
    verbose=True,
)
display(knn_cv_df)

knn_model_cfg = {k: v for k, v in best_knn_config.items() if k != "prob_thres"}
knn_result = train_and_evaluate(
    KNN(**knn_model_cfg),
    split_data,
    model_name="KNN",
    config=best_knn_config,
    X_test=prep.X_test,
    fit_orig_only=True,
    plot_cm=True,
)
evaluation_results.append(knn_result)


### Model 2: Naive Bayes (Weighted NB Scratch)

Naive Bayes tidak memakai pipeline scaling/SMOTE/PCA karena ketiganya merusak asumsi independensi dan distribusi fitur NB. Fitur dibuat dengan `prepare_nb_features`, lalu di dalam `WideNaiveBayes` dilakukan diskritisasi MDL, fitur gabungan, dan pembobotan per fitur. Detail eksperimen ada di `notebooks/naive_bayes_experiment.ipynb`.

Submisi NB yang sesuai spesifikasi adalah `output/submission_wnb.csv` (public score 0.85167) yang dibuat di `notebooks/naive_bayes_experiment.ipynb`.


In [ ]:
nb_split = split_train_validation(
    prepare_nb_features(train_df, extra=True, flags=True).values,
    train_df['is_canceled'].values,
    test_size=0.15,
    random_state=RANDOM_STATE,
    verbose=True,
)
X_test_nb = prepare_nb_features(test_df, extra=True, flags=True).values

best_nb_config, nb_cv_df = cross_validate_configs(
    WideNaiveBayes,
    DEFAULT_NB_CONFIGS,
    nb_split.X_tr,
    nb_split.y_tr,
    k_folds=5,
    random_state=RANDOM_STATE,
    model_name="Naive Bayes",
    verbose=True,
)
display(nb_cv_df)

nb_result = train_and_evaluate(
    WideNaiveBayes(**best_nb_config),
    nb_split,
    model_name="Naive Bayes",
    config=best_nb_config,
    X_test=X_test_nb,
    plot_cm=True,
)
evaluation_results.append(nb_result)


### Model 3: Decision Tree Learning (C4.5 Scratch)

Fitur DTL dibuat dengan prepare_dtl_features. Kolom kategori tetap diperlakukan
sebagai kategori oleh pohon. Eksperimen serta pembanding scikit-learn ada
di notebooks/dtl_experiment.ipynb.


In [ ]:
X_dtl = prepare_dtl_features(train_df.drop_duplicates())
X_dtl_test = prepare_dtl_features(test_df)
dtl_categorical_indices = [
    X_dtl.columns.get_loc(column) for column in DTL_CATEGORICAL_COLUMNS
]
dtl_configs = [
    dict(config, categorical_features=dtl_categorical_indices)
    for config in DEFAULT_DTL_CONFIGS
]
dtl_split = split_train_validation(
    X_dtl.values,
    train_df.drop_duplicates()["is_canceled"].values,
    test_size=0.15,
    random_state=RANDOM_STATE,
)

best_dtl_config, dtl_cv_df = cross_validate_configs(
    C45DecisionTree,
    dtl_configs,
    dtl_split.X_tr,
    dtl_split.y_tr,
    k_folds=5,
    random_state=RANDOM_STATE,
    model_name="DTL",
)
display(dtl_cv_df)

dtl_result = train_and_evaluate(
    C45DecisionTree(**best_dtl_config),
    dtl_split,
    model_name="DTL",
    config=best_dtl_config,
    X_test=X_dtl_test.values,
)
evaluation_results.append(dtl_result)


### Model 4: Logistic Regression (Scratch)

Implementasi `LogisticRegression` di `src/models/logistic_regression.py` mewarisi `BaseModel` dengan pilihan optimasi Stochastic Gradient Ascent (SGA) dan Adam serta regularisasi L1, L2, dan Elastic Net. Perbandingan dengan `scikit-learn` dan detail eksperimen tersedia di `notebooks/logistic_regression_experiment.ipynb`.

In [ ]:
best_logreg_config, logreg_cv_df = cross_validate_configs(
    LogisticRegression,
    DEFAULT_LOGREG_CONFIGS,
    split_data.X_tr,
    split_data.y_tr,
    k_folds=5,
    random_state=RANDOM_STATE,
    model_name="Logistic Regression",
    verbose=True,
)
display(logreg_cv_df)

logreg_model_cfg = {k: v for k, v in best_logreg_config.items() if k != "prob_thres"}
logreg_result = train_and_evaluate(
    LogisticRegression(**logreg_model_cfg),
    split_data,
    model_name="Logistic Regression",
    config=best_logreg_config,
    X_test=prep.X_test,
    fit_orig_only=True,
    plot_cm=True,
)
evaluation_results.append(logreg_result)

### Model Comparison


In [ ]:
comparison_df = compare_models(evaluation_results)
display(comparison_df)

best_result = max(evaluation_results, key=lambda item: item["macro_f1"])
print(f"Best model: {best_result['model_name']} (Macro F1 = {best_result['macro_f1']:.5f})")


## Submission


In [ ]:
best_slug = best_result["model_name"].lower().replace(" ", "_")
OUTPUT_DIR = Path("output") if Path("output").is_dir() else Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_filename = str(OUTPUT_DIR / f"{best_slug}.pkl")
submission_csv_path = str(OUTPUT_DIR / f"submission_{best_slug}.csv")
best_X_test = (
    best_result["X_test"]
    if best_result.get("X_test") is not None
    else prep.X_test
)

loaded_best_model = save_and_verify_model(best_result["model"], model_filename, verbose=True)
best_prob_thres = best_result.get("config", {}).get("prob_thres")
submission = generate_submission(
    loaded_best_model,
    best_X_test,
    test_df["id"],
    sample_submission,
    submission_csv_path,
    prob_threshold=best_prob_thres,
    random_state=RANDOM_STATE,
    verbose=True,
)
display(submission.head())
